# Evaluation: Base vs. Fine-Tuned Text-to-SQL

Compare base Qwen3-8B against the GRPO fine-tuned model across:

1. **BIRD Dev** — 673 text-to-SQL examples with execution accuracy (EX)
2. **NNDSS Custom** — 50 held-out domain-specific pairs against live Trino
3. **General Capability** — EvalHub `leaderboard-v2` benchmarks (regression check)
4. **Performance** — EvalHub `guidellm` throughput/latency benchmarks

## Setup

In [ ]:
%env UV_EXTRA_INDEX_URL=https://pypi.org/simple
!uv pip install pandas torch peft safetensors

In [ ]:
import glob
import json
import os
import sys
from pathlib import Path

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

sys.path.insert(0, str(Path("../").resolve()))
from reward.grader import grade
from reward.sql_executor import execute_sqlite

In [ ]:
# Paths — adjust to your PVC mount or local paths
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")
PVC_MOUNT_PATH = "/opt/app-root/src/shared"

MODEL_PATH = "Qwen/Qwen3-8B"
GRPO_CKPT_DIR = f"{PVC_MOUNT_PATH}/text2sql/grpo_output"
BIRD_DB_ROOT = f"{PVC_MOUNT_PATH}/text2sql/bird_databases"

# Trino for NNDSS eval
TRINO_HOST = "localhost"  # via port-forward
TRINO_PORT = 8090

## Load Models

In [ ]:
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model.eval()
print("Base model loaded")

In [ ]:
print("Loading fine-tuned model (merging LoRA adapter from verl checkpoint)...")

import collections
from torch.distributed.tensor import DTensor

CKPT_BASE = os.path.join(GRPO_CKPT_DIR, "checkpoints")
latest_file = os.path.join(CKPT_BASE, "latest_checkpointed_iteration.txt")
with open(latest_file) as f:
    latest_step = f.read().strip()
actor_dir = os.path.join(CKPT_BASE, f"global_step_{latest_step}", "actor")
print(f"Using verl checkpoint: global_step_{latest_step}")

# Load rank-0 shard (verl stores full LoRA copy per rank)
shard_files = sorted(glob.glob(os.path.join(actor_dir, "model_world_size_*_rank_0.pt")))
state_dict = torch.load(shard_files[0], map_location="cpu", weights_only=False)

# Extract LoRA keys — verl wraps weights as DTensor (FSDP), so unwrap
# to plain tensors via _local_tensor to avoid DeviceMesh dispatch.
lora_state = {}
for k, v in state_dict.items():
    if "lora_" not in k:
        continue
    if isinstance(v, DTensor):
        lora_state[k] = v._local_tensor.clone()
    else:
        lora_state[k] = v.clone()
del state_dict
print(f"  LoRA parameters: {len(lora_state)} tensors")

# Read LoRA config from verl metadata
with open(os.path.join(actor_dir, "lora_train_meta.json")) as f:
    lora_meta = json.load(f)

# Build a temporary PEFT adapter directory
adapter_dir = os.path.join(GRPO_CKPT_DIR, "_merged_adapter")
os.makedirs(adapter_dir, exist_ok=True)

# Infer target modules from LoRA key names
target_modules = sorted({
    k.split("lora_")[0].rstrip(".")
      .replace("base_model.model.", "")
      .rsplit(".", 1)[-1]
    for k in lora_state
})
adapter_config = {
    "peft_type": "LORA",
    "auto_mapping": None,
    "base_model_name_or_path": MODEL_PATH,
    "bias": "none",
    "fan_in_fan_out": False,
    "inference_mode": True,
    "init_lora_weights": True,
    "layers_to_transform": None,
    "layers_pattern": None,
    "lora_alpha": lora_meta["lora_alpha"],
    "lora_dropout": 0.0,
    "modules_to_save": None,
    "r": lora_meta["r"],
    "revision": None,
    "target_modules": target_modules,
    "task_type": lora_meta.get("task_type", "CAUSAL_LM"),
}
with open(os.path.join(adapter_dir, "adapter_config.json"), "w") as f:
    json.dump(adapter_config, f, indent=2)

# Save LoRA weights as safetensors
from safetensors.torch import save_file
save_file(lora_state, os.path.join(adapter_dir, "adapter_model.safetensors"))
del lora_state
print(f"  Merged adapter written to {adapter_dir}")

# Load
ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="auto",
)
ft_model = PeftModel.from_pretrained(ft_base, adapter_dir)
ft_model = ft_model.merge_and_unload()
ft_model.eval()
print("Fine-tuned model loaded")

In [ ]:
def generate_sql(model, messages, max_tokens=256):
    """Generate SQL from a model given chat messages."""
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens,
            temperature=0.1, do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True,
    )
    return response.strip()


def extract_sql_from_response(response):
    """Extract SQL from model response."""
    import re
    text = re.sub(r"<think>.*?</think>", "", response, flags=re.DOTALL).strip()
    match = re.search(r"```(?:sql)?\s*\n?(.*?)```", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    match = re.search(r"(SELECT\s+.+?)(?:;|\Z)", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return text

## A — BIRD Dev Evaluation

Evaluate on 673 BIRD mini-Dev examples using execution accuracy (EX).
Generated SQL is executed against the BIRD SQLite databases and
results are compared to gold SQL using ReViSQL's grading logic.

In [ ]:
import shutil, subprocess, zipfile

db_count = len(glob.glob(os.path.join(BIRD_DB_ROOT, "*/*.sqlite")))
if db_count >= 11:
    print(f"BIRD databases present: {db_count} databases in {BIRD_DB_ROOT}")
else:
    print(f"Only {db_count} BIRD databases found — downloading from bird-bench...")
    dl_dir = "/tmp/bird_dl"
    zip_path = "/tmp/minidev.zip"
    os.makedirs(dl_dir, exist_ok=True)
    subprocess.run(
        ["curl", "-L", "-o", zip_path,
         "https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip"],
        check=True,
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dl_dir)
    src = os.path.join(dl_dir, "minidev", "MINIDEV", "dev_databases")
    os.makedirs(BIRD_DB_ROOT, exist_ok=True)
    for db_dir in Path(src).iterdir():
        dest = Path(BIRD_DB_ROOT) / db_dir.name
        if not dest.exists():
            shutil.copytree(db_dir, dest)
    shutil.rmtree(dl_dir, ignore_errors=True)
    os.remove(zip_path)
    db_count = len(glob.glob(os.path.join(BIRD_DB_ROOT, "*/*.sqlite")))
    print(f"Downloaded {db_count} BIRD databases to {BIRD_DB_ROOT}")

In [ ]:
import pyarrow.parquet as pq

bird_eval = pq.read_table(os.path.join(DATA_DIR, "bird_eval.parquet")).to_pandas()
print(f"BIRD eval set: {len(bird_eval)} examples")

# Limit for initial testing (set to len(bird_eval) for full eval)
BIRD_EVAL_LIMIT = 50
bird_eval_subset = bird_eval.head(BIRD_EVAL_LIMIT)
print(f"Evaluating {len(bird_eval_subset)} examples")

In [ ]:
def evaluate_bird(model, eval_df, db_root):
    """Run BIRD execution accuracy evaluation."""
    results = []
    for idx, row in eval_df.iterrows():
        messages = list(row["prompt"])
        gold_sql = row["reward_spec"]["ground_truth"]
        method = row["reward_spec"].get("method", "set")
        db_id = row["db_id"]
        db_file = os.path.join(db_root, db_id, f"{db_id}.sqlite")

        if not os.path.exists(db_file):
            raise FileNotFoundError(
                f"BIRD database missing: {db_file} — run the download cell above"
            )

        response = generate_sql(model, messages)
        pred_sql = extract_sql_from_response(response)

        correct = False
        gold_res, gold_err = execute_sqlite(gold_sql, db_file)
        pred_res, pred_err = execute_sqlite(pred_sql, db_file)
        if gold_res is not None and pred_res is not None:
            correct, _ = grade(list(gold_res), list(pred_res), method)

        results.append({
            "db_id": db_id, "correct": correct,
            "gold_sql": gold_sql[:100], "pred_sql": pred_sql[:100],
        })
        if (idx + 1) % 10 == 0:
            acc = sum(r["correct"] for r in results) / len(results)
            print(f"  [{idx+1}/{len(eval_df)}] Running EX: {acc:.1%}")

    accuracy = sum(r["correct"] for r in results) / len(results)
    return accuracy, results


print("Evaluating base model on BIRD dev...")
base_bird_acc, base_bird_results = evaluate_bird(base_model, bird_eval_subset, BIRD_DB_ROOT)
print(f"Base model BIRD EX: {base_bird_acc:.1%}\n")

print("Evaluating fine-tuned model on BIRD dev...")
ft_bird_acc, ft_bird_results = evaluate_bird(ft_model, bird_eval_subset, BIRD_DB_ROOT)
print(f"Fine-tuned model BIRD EX: {ft_bird_acc:.1%}")

## B — NNDSS Custom Evaluation

Evaluate on held-out NNDSS question-SQL pairs against live Trino.

In [ ]:
from reward.nndss_schema import NNDSS_DDL
from reward.sql_executor import execute_trino

nndss_eval_path = Path("../data/nndss_pairs/nndss_pairs.jsonl")
with open(nndss_eval_path) as f:
    all_nndss = [json.loads(line) for line in f]

# Use last 50 as held-out eval
nndss_eval = all_nndss[-50:]
print(f"NNDSS eval set: {len(nndss_eval)} pairs")

In [ ]:
def evaluate_nndss(model, eval_pairs, trino_host, trino_port):
    """Run NNDSS execution accuracy evaluation."""
    results = []
    for pair in eval_pairs:
        messages = [
            {"role": "system", "content": "You are a SQL expert. Write a SQL query that answers the question. Output only the SQL."},
            {"role": "user", "content": f"Schema:\n{NNDSS_DDL}\n\nQuestion: {pair['question']}"},
        ]
        response = generate_sql(model, messages)
        pred_sql = extract_sql_from_response(response)

        gold_res, gold_err = execute_trino(pair["sql"], trino_host, trino_port)
        pred_res, pred_err = execute_trino(pred_sql, trino_host, trino_port)

        correct = False
        if gold_res is not None and pred_res is not None:
            correct, _ = grade(list(gold_res), list(pred_res), "set")

        results.append({
            "pattern": pair.get("pattern", "unknown"),
            "correct": correct,
            "question": pair["question"][:80],
        })

    accuracy = sum(r["correct"] for r in results) / len(results) if results else 0
    return accuracy, results


print("Evaluating base model on NNDSS...")
base_nndss_acc, base_nndss_results = evaluate_nndss(base_model, nndss_eval, TRINO_HOST, TRINO_PORT)
print(f"Base model NNDSS EX: {base_nndss_acc:.1%}\n")

print("Evaluating fine-tuned model on NNDSS...")
ft_nndss_acc, ft_nndss_results = evaluate_nndss(ft_model, nndss_eval, TRINO_HOST, TRINO_PORT)
print(f"Fine-tuned model NNDSS EX: {ft_nndss_acc:.1%}")

In [ ]:
# Per-category breakdown
from collections import defaultdict

def category_breakdown(results):
    by_cat = defaultdict(list)
    for r in results:
        by_cat[r["pattern"]].append(r["correct"])
    return {cat: sum(v)/len(v) for cat, v in sorted(by_cat.items())}

print("\nNNDSS per-category accuracy:")
print(f"{'Category':<25} {'Base':>8} {'Fine-tuned':>12}")
print("-" * 48)
base_cats = category_breakdown(base_nndss_results)
ft_cats = category_breakdown(ft_nndss_results)
for cat in sorted(set(list(base_cats.keys()) + list(ft_cats.keys()))):
    b = base_cats.get(cat, 0)
    f = ft_cats.get(cat, 0)
    print(f"{cat:<25} {b:>7.0%} {f:>11.0%}")

## C — General Capability via EvalHub

Submit `lm-evaluation-harness` jobs to EvalHub for `leaderboard-v2`
benchmarks (MMLU, ARC, HellaSwag, IFeval). This verifies fine-tuning
hasn't degraded general capabilities.

Requires the LLMInferenceService to be deployed (see notebook 04).

In [ ]:
# Submit EvalHub evaluation jobs via the RHOAI API
# Adjust the endpoint and model names for your deployment

EVALHUB_NAMESPACE = "redhat-ods-applications"
BASE_MODEL_ENDPOINT = "<BASE_MODEL_ENDPOINT>"  # e.g., deployed base Qwen3-8B
FT_MODEL_ENDPOINT = "<FT_MODEL_ENDPOINT>"  # LLMInferenceService endpoint

evalhub_base_job = f"""\
apiVersion: trustyai.opendatahub.io/v1alpha1
kind: EvalJob
metadata:
  name: text2sql-base-leaderboard
  namespace: {EVALHUB_NAMESPACE}
spec:
  provider: lm-evaluation-harness
  collection: leaderboard-v2
  model:
    url: {BASE_MODEL_ENDPOINT}
    name: qwen3-8b-base
"""

evalhub_ft_job = f"""\
apiVersion: trustyai.opendatahub.io/v1alpha1
kind: EvalJob
metadata:
  name: text2sql-ft-leaderboard
  namespace: {EVALHUB_NAMESPACE}
spec:
  provider: lm-evaluation-harness
  collection: leaderboard-v2
  model:
    url: {FT_MODEL_ENDPOINT}
    name: qwen3-8b-text2sql
"""

print("EvalHub job manifests ready.")
print("Apply with: oc apply -f <manifest>")
print("Results will be tracked in MLflow.")

## Results Summary

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# BIRD + NNDSS comparison
benchmarks = ["BIRD Dev (EX)", "NNDSS (EX)"]
base_scores = [base_bird_acc, base_nndss_acc]
ft_scores = [ft_bird_acc, ft_nndss_acc]

x = range(len(benchmarks))
width = 0.35
axes[0].bar([i - width/2 for i in x], base_scores, width, label="Base Qwen3-8B", color="#4A90D9")
axes[0].bar([i + width/2 for i in x], ft_scores, width, label="Fine-tuned", color="#EE0000")
axes[0].set_ylabel("Execution Accuracy")
axes[0].set_title("Text-to-SQL Benchmarks")
axes[0].set_xticks(x)
axes[0].set_xticklabels(benchmarks)
axes[0].legend()
axes[0].set_ylim(0, 1)

# NNDSS per-category
categories = sorted(ft_cats.keys())
base_cat_scores = [base_cats.get(c, 0) for c in categories]
ft_cat_scores = [ft_cats.get(c, 0) for c in categories]

x2 = range(len(categories))
axes[1].barh([i - width/2 for i in x2], base_cat_scores, width, label="Base", color="#4A90D9")
axes[1].barh([i + width/2 for i in x2], ft_cat_scores, width, label="Fine-tuned", color="#EE0000")
axes[1].set_xlabel("Execution Accuracy")
axes[1].set_title("NNDSS by SQL Pattern")
axes[1].set_yticks(x2)
axes[1].set_yticklabels(categories, fontsize=8)
axes[1].legend()
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

print(f"\n{'Benchmark':<20} {'Base':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 52)
print(f"{'BIRD Dev (EX)':<20} {base_bird_acc:>9.1%} {ft_bird_acc:>11.1%} {ft_bird_acc - base_bird_acc:>+7.1%}")
print(f"{'NNDSS (EX)':<20} {base_nndss_acc:>9.1%} {ft_nndss_acc:>11.1%} {ft_nndss_acc - base_nndss_acc:>+7.1%}")